In [0]:
%sql
SHOW TABLES IN healthcare_demo.silver;

database,tableName,isTemporary
silver,patient_referral_details,false


In [0]:
from pyspark.sql import functions as F

silver = (
    spark.table("healthcare_demo.silver.patient_referral_details")
    .withColumn(
        "waiting_days",
        F.datediff(F.current_date(), F.col("referral_date"))
    )
)

display(silver)

location_id,specialty,referral_id,patient_id,age_band,area_code,referral_date,priority,referral_status,appointment_id,appointment_date,appointment_status,location_name,available_slots,_silver_processed_at,waiting_days
L002,Orthopaedics,R0007,P0007,31-45,NG2,2026-08-23,Urgent,Open,A0007,2026-12-01,Cancelled,North Clinic,8,2026-11-09T21:09:28.250Z,79
L002,Cardiology,R0009,P0009,61-75,NG2,2026-09-10,Routine,Open,A0009,2026-11-24,Cancelled,null,null,2026-11-09T21:09:28.250Z,61
L002,Orthopaedics,R0059,P0059,46-60,NG2,2026-07-25,Urgent,Open,A0059,2026-12-21,Booked,North Clinic,8,2026-11-09T21:09:28.250Z,108
L002,Dermatology,R0061,P0061,31-45,NG1,2026-05-24,Urgent,Open,A0061,2026-11-16,Attended,null,null,2026-11-09T21:09:28.250Z,170
L003,Orthopaedics,R0072,P0072,31-45,NG5,2026-05-18,Urgent,Open,A0072,2026-12-19,Attended,null,null,2026-11-09T21:09:28.250Z,176
L002,Dermatology,R0085,P0085,31-45,NG7,2026-09-22,Routine,Open,A0085,2026-11-15,Booked,null,null,2026-11-09T21:09:28.250Z,49
L002,Cardiology,R0088,P0088,18-30,NG1,2026-09-08,Routine,Open,A0088,2026-12-03,Cancelled,null,null,2026-11-09T21:09:28.250Z,63
L001,Orthopaedics,R0005,P0005,61-75,NG1,2026-09-23,Routine,Open,A0005,2026-12-08,Cancelled,null,null,2026-11-09T21:09:28.250Z,48
L001,Cardiology,R0008,P0008,18-30,NG2,2026-09-26,Urgent,Open,A0008,2026-11-15,Booked,Central Clinic,5,2026-11-09T21:09:28.250Z,45
L002,Orthopaedics,R0011,P0011,18-30,NG2,2026-06-08,Routine,Open,A0011,2026-11-14,Attended,North Clinic,8,2026-11-09T21:09:28.250Z,155


In [0]:
from pyspark.sql import functions as F

silver = (
    spark.table("healthcare_demo.silver.patient_referral_details")
    .withColumn(
        "waiting_days",
        F.datediff(F.current_date(), F.col("referral_date"))
    )
)

waiting_report = (
    silver
    .groupBy("specialty", "priority")
    .agg(
        F.countDistinct("referral_id").alias("total_referrals"),
        F.round(F.avg("waiting_days"), 1).alias("average_waiting_days"),
        F.max("waiting_days").alias("maximum_waiting_days")
    )
    .orderBy("specialty", "priority")
)

display(waiting_report)

specialty,priority,total_referrals,average_waiting_days,maximum_waiting_days
Cardiology,Routine,19,84.9,175
Cardiology,Urgent,18,98.7,178
Dermatology,Routine,17,81.2,179
Dermatology,Urgent,17,115.2,179
Orthopaedics,Routine,17,92.2,155
Orthopaedics,Urgent,12,104.7,176


In [0]:
waiting_report.write.format("delta").mode("overwrite").saveAsTable(
    "healthcare_demo.gold.waiting_time_summary"
)

In [0]:
from pyspark.sql import functions as F

appointments = spark.table(
    "healthcare_demo.bronze.appointments"
)

capacity = spark.table(
    "healthcare_demo.bronze.capacity"
)

silver = spark.table(
    "healthcare_demo.silver.patient_referral_details"
)

In [0]:
referral_appointments = (
    appointments
    .join(
        silver.select("referral_id", "specialty"),
        on="referral_id",
        how="left"
    )
)

display(referral_appointments)

referral_id,appointment_id,appointment_date,location_id,appointment_status,_ingested_at,specialty
R0001,A0001,2026-12-27,L002,Attended,2026-11-10T18:09:20.846Z,Dermatology
R0002,A0002,2026-11-26,L001,Cancelled,2026-11-10T18:09:20.846Z,Cardiology
R0003,A0003,2026-11-27,L003,Attended,2026-11-10T18:09:20.846Z,Orthopaedics
R0004,A0004,2026-12-20,L001,Attended,2026-11-10T18:09:20.846Z,Orthopaedics
R0005,A0005,2026-12-08,L001,Cancelled,2026-11-10T18:09:20.846Z,Orthopaedics
R0006,A0006,2026-11-28,L001,Attended,2026-11-10T18:09:20.846Z,Orthopaedics
R0007,A0007,2026-12-01,L002,Cancelled,2026-11-10T18:09:20.846Z,Orthopaedics
R0008,A0008,2026-11-15,L001,Booked,2026-11-10T18:09:20.846Z,Cardiology
R0009,A0009,2026-11-24,L002,Cancelled,2026-11-10T18:09:20.846Z,Cardiology
R0010,A0010,2026-11-19,L003,Booked,2026-11-10T18:09:20.846Z,Cardiology


In [0]:
location_mapping = (
    capacity
    .select("location_id", "location_name")
    .distinct()
)

capacity_details = capacity.select(
    "location_id",
    "specialty",
    "available_slots"
)

capacity_enriched = (
    referral_appointments
    .join(
        location_mapping,
        on="location_id",
        how="left"
    )
    .join(
        capacity_details,
        on=["location_id", "specialty"],
        how="left"
    )
)

display(capacity_enriched)

location_id,specialty,referral_id,appointment_id,appointment_date,appointment_status,_ingested_at,location_name,available_slots
L002,Dermatology,R0001,A0001,2026-12-27,Attended,2026-11-10T18:09:20.846Z,North Clinic,null
L001,Cardiology,R0002,A0002,2026-11-26,Cancelled,2026-11-10T18:09:20.846Z,Central Clinic,5
L003,Orthopaedics,R0003,A0003,2026-11-27,Attended,2026-11-10T18:09:20.846Z,South Clinic,null
L001,Orthopaedics,R0004,A0004,2026-12-20,Attended,2026-11-10T18:09:20.846Z,Central Clinic,null
L001,Orthopaedics,R0005,A0005,2026-12-08,Cancelled,2026-11-10T18:09:20.846Z,Central Clinic,null
L001,Orthopaedics,R0006,A0006,2026-11-28,Attended,2026-11-10T18:09:20.846Z,Central Clinic,null
L002,Orthopaedics,R0007,A0007,2026-12-01,Cancelled,2026-11-10T18:09:20.846Z,North Clinic,8
L001,Cardiology,R0008,A0008,2026-11-15,Booked,2026-11-10T18:09:20.846Z,Central Clinic,5
L002,Cardiology,R0009,A0009,2026-11-24,Cancelled,2026-11-10T18:09:20.846Z,North Clinic,null
L003,Cardiology,R0010,A0010,2026-11-19,Booked,2026-11-10T18:09:20.846Z,South Clinic,null


In [0]:
capacity_report = (
    capacity_enriched
    .groupBy(
        "location_id",
        "location_name",
        "specialty"
    )
    .agg(
        F.countDistinct("appointment_id").alias("total_appointments"),
        F.max("available_slots").alias("available_slots")
    )
    .orderBy("location_id", "specialty")
)

display(capacity_report)

location_id,location_name,specialty,total_appointments,available_slots
L001,Central Clinic,Cardiology,15,5
L001,Central Clinic,Dermatology,7,null
L001,Central Clinic,Orthopaedics,7,null
L002,North Clinic,Cardiology,13,null
L002,North Clinic,Dermatology,13,null
L002,North Clinic,Orthopaedics,16,8
L003,South Clinic,Cardiology,9,null
L003,South Clinic,Dermatology,14,6
L003,South Clinic,Orthopaedics,6,null


In [0]:
capacity_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_demo.gold.clinic_capacity_summary")

print("Gold clinic capacity summary saved successfully")

Gold clinic capacity summary saved successfully


In [0]:
display(
    appointments.orderBy("appointment_date").limit(20)
)

appointment_id,referral_id,appointment_date,location_id,appointment_status,_ingested_at
A0079,R0079,2026-11-10,L003,Booked,2026-11-10T18:09:20.846Z
A0016,R0016,2026-11-11,L003,Attended,2026-11-10T18:09:20.846Z
A0036,R0036,2026-11-11,L003,Cancelled,2026-11-10T18:09:20.846Z
A0039,R0039,2026-11-12,L001,Booked,2026-11-10T18:09:20.846Z
A0031,R0031,2026-11-13,L002,Attended,2026-11-10T18:09:20.846Z
A0037,R0037,2026-11-13,L002,Booked,2026-11-10T18:09:20.846Z
A0011,R0011,2026-11-14,L002,Attended,2026-11-10T18:09:20.846Z
A0029,R0029,2026-11-15,L003,Attended,2026-11-10T18:09:20.846Z
A0085,R0085,2026-11-15,L002,Booked,2026-11-10T18:09:20.846Z
A0080,R0080,2026-11-15,L001,Cancelled,2026-11-10T18:09:20.846Z


In [0]:
display(capacity)

location_id,location_name,specialty,available_slots,_ingested_at
L001,Central Clinic,Cardiology,5,2026-11-10T18:09:24.855Z
L002,North Clinic,Orthopaedics,8,2026-11-10T18:09:24.855Z
L003,South Clinic,Dermatology,6,2026-11-10T18:09:24.855Z


In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)

audit_schema = StructType([
    StructField("check_name", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("failed_records", IntegerType(), False),
    StructField("result", StringType(), False),
    StructField("checked_at", TimestampType(), False)
])

empty_audit = spark.createDataFrame([], audit_schema)

empty_audit.write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable("healthcare_demo.gold.data_quality_audit")

print("Data quality audit table created")

Data quality audit table created
